<a href="https://colab.research.google.com/github/GuruShrihari/Credit_Card_Fraud/blob/main/Credit_Card.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, precision_recall_curve
from imblearn.pipeline import Pipeline

In [ ]:
d = pd.read_csv("creditcard.csv")

print("Max nulls in any column:", d.isnull().sum().max())

df = d.copy()

#Extremely imbalanced , not efficient to over/undersample

print(df["Class"].value_counts())

In [ ]:
for labels in df.columns:
  plt.hist(df[df["Class"] == 1][labels],color="blue", label="Fraud",alpha= 0.7, density=True)
  plt.hist(df[df["Class"] == 0][labels],color="red", label="NotFraud",alpha= 0.7, density=True)
  plt.title(labels)
  plt.xlabel(labels)
  plt.ylabel("Frequency")
  plt.legend()
  plt.show()

# **Train, Test , Validation**

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,   #makes sure that train and test sets keep the same class proportions as the original data.
    random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=42
)

In [ ]:
print(y_train.value_counts())

# **Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("oversample", RandomOverSampler(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_valid_pred = pipeline.predict(X_valid)

y_valid_proba = pipeline.predict_proba(X_valid)[:, 1]

print("Validation Results:")
print(classification_report(y_valid, y_valid_pred))


In [ ]:
print("Validation ROC-AUC:", roc_auc_score(y_valid, y_valid_proba))
print("Validation PR-AUC:", average_precision_score(y_valid, y_valid_proba))

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_valid, y_valid_proba)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

threshold = 0.1  # lower threshold => catchs more frauds (Threshold is set lower as the dataset is imbalanced)
y_valid_custom = (y_valid_proba >= threshold).astype(int)

print(f"Validation Results at the threshold={threshold}")
print(classification_report(y_valid, y_valid_custom))

In [ ]:
y_test_proba = pipeline.predict_proba(X_test)[:, 1]


y_test_custom = (y_test_proba >= threshold).astype(int)

print("Test Results:")
print(classification_report(y_test, y_test_custom))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("Test PR-AUC:", average_precision_score(y_test, y_test_proba))